# Notebook 01: Data Acquisition

Builds the evaluation corpora for the TPC benchmark:
- **Positive corpus**: retracted papers from PubMed
- **Negative corpus**: clean papers matched by domain and year
- **Synthetic corpus**: WordNet and back-translation paraphrases

Addresses paper Section 5.1 (Methodology: Data Acquisition).

In [1]:
import pandas as pd
from pathlib import Path
from tpc.acquisition.pubmed import fetch_retracted_abstracts, fetch_clean_abstracts
from tpc.acquisition.synthetic import generate_from_registry, generate_wordnet

DATA_DIR = Path('../data/ground_truth')
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# 1. Positive corpus: retracted papers
print('Fetching retracted papers...')
positive = fetch_retracted_abstracts(
    max_results=5000,
    reasons=['paper mill', 'tortured phrases', 'manipulation'],
)
df_pos = pd.DataFrame(positive)
print(f'Retrieved: {len(df_pos)} retracted abstracts')
df_pos.head(3)

Fetching retracted papers...
Retrieved: 10 retracted abstracts


,pmid,title,abstract,journal,year,label,source
0,37843654,Electronic and optical structural manipulation...,Monolayer NbS,Journal of molecular modeling,2023,retracted,pubmed
1,36247705,Systematic Review and Meta-Analysis of the Eva...,With the accelerated pace of life in modern so...,Emergency medicine international,2022,retracted,pubmed
2,34721644,Observation on the Curative Effect of Massage ...,"In this prospective study, we used the random ...",Evidence-based complementary and alternative m...,2021,retracted,pubmed


In [3]:
# 2. Negative corpus: clean papers
print('Fetching clean papers...')
negative = fetch_clean_abstracts(max_results=5000)
df_neg = pd.DataFrame(negative)
print(f'Retrieved: {len(df_neg)} clean abstracts')

# Save both
df_pos.to_csv(DATA_DIR / 'confirmed_tortured.tsv', sep='\t', index=False)
df_neg.to_csv(DATA_DIR / 'confirmed_clean.tsv', sep='\t', index=False)
print('Saved corpora.')

Fetching clean papers...
Retrieved: 4974 clean abstracts
Saved corpora.


In [4]:
# 3. Corpus statistics (Appendix B)
stats = pd.DataFrame([
    {'corpus': 'Positive (retracted)', 'n': len(df_pos), 
     'mean_length': df_pos['abstract'].str.len().mean()},
    {'corpus': 'Negative (clean)',     'n': len(df_neg), 
     'mean_length': df_neg['abstract'].str.len().mean()},
])
print(stats)

                 corpus     n  mean_length
0  Positive (retracted)    10   1072.80000
1      Negative (clean)  4974    805.07881


In [5]:
# 4. Synthetic corpus: registry substitution
clean_sentences = df_neg['abstract'].dropna().tolist()
synthetic_reg   = generate_from_registry(clean_sentences, n_samples=10000)
print(f'Registry substitution: {len(synthetic_reg)} examples')

synthetic_wn = generate_wordnet(clean_sentences, substitution_rate=0.15)
print(f'WordNet substitution:  {len(synthetic_wn)} examples')

df_synth = pd.DataFrame(synthetic_reg + synthetic_wn)
df_synth.to_csv(DATA_DIR / 'synthetic_corpus.tsv', sep='\t', index=False)
print('Saved synthetic corpus.')

Registry substitution: 30 examples
WordNet substitution:  4657 examples
Saved synthetic corpus.
